# Steganography GAN — train on a cloud GPU (Colab / similar)

**What this does:** installs a CUDA build of PyTorch, fetches this project, runs  
`scripts/train_production_gan_gpu.py` (the **app-compatible** trainer — not the old `colab_train_gpu.py` at repo root).

1. In Colab: **Runtime → Change runtime type → GPU** (T4 is fine; A100 is faster if available).
2. If your project is only on your laptop: use **File → Upload** to your Drive (see next cells) or push to GitHub and set `REPO_URL` below.
3. Checkpoints are saved to `models/checkpoints/*/best_model.pth` under the repo. Copy with Google Drive (optional cell) or **download** from the Files pane.
4. If **CUDA is still `False` after the install cell:** **Runtime → Restart session**, then run only from the install cell down (Colab’s default CPU-only torch is replaced on restart).

In [ ]:
# 1) Check GPU and install PyTorch (CUDA) + minimal deps for this project
import subprocess, sys, os

def sh(cmd: str) -> int:
    print("$", cmd)
    return subprocess.call(cmd, shell=True)

try:
    import torch
    print("torch before install:", getattr(torch, "__version__", "?"))
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print(e)

# Colab: replace index URL if Colab’s default CUDA outpaces this wheel (see pytorch.org get-started)
INSTALL = "pip install -q 'torch' 'torchvision' 'torchaudio' --index-url https://download.pytorch.org/whl/cu124"
sh(INSTALL + " 2>&1 | tail -5 || true")

import torch  # re-import after install in fresh sessions
import importlib, sys
if "torch" in sys.modules:
    importlib.reload(torch)  # best-effort; restart runtime if still wrong

sh("pip install -q cryptography pillow numpy scipy")
print("cuda available:", torch.cuda.is_available(), "|", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Get the project: choose ONE of (A) git clone, (B) Google Drive, (C) Colab file upload to /content

import os, subprocess, zipfile, shutil, pathlib
WORKDIR = "/content/stego_project"  # change if you use another path

# (A) Public git repo (replace with your URL). Leave empty to skip.
REPO_URL = ""  # e.g. "https://github.com/<you>/steganography-project.git"

if REPO_URL.strip() and not os.path.isdir(pathlib.Path(WORKDIR) / ".git"):
    p = pathlib.Path(WORKDIR).parent
    p.mkdir(parents=True, exist_ok=True)
    if os.path.isdir(WORKDIR):
        shutil.rmtree(WORKDIR)
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, WORKDIR])
    print("Cloned to:", WORKDIR)
elif os.path.isdir(pathlib.Path(WORKDIR) / ".git") or os.path.isfile(pathlib.Path(WORKDIR) / "scripts" / "train_production_gan_gpu.py"):
    print("Using existing project at:", WORKDIR)
else:
    # (B) After uploading a zip to Drive, set ZIP_PATH, or (C) upload a zip in Colab file browser to /content and set name
    DRIVE_ZIP = ""  # e.g. "/content/drive/MyDrive/steganography-project.zip" after mounting Drive (next cell)
    UPLOADED = "/content/steganography-project.zip"  # or upload the repo zip to /content and keep this name
    for candidate in (DRIVE_ZIP, UPLOADED):
        if candidate and os.path.isfile(candidate):
            extract_to = str(pathlib.Path(WORKDIR).parent)
            with zipfile.ZipFile(candidate, "r") as z:
                z.extractall(extract_to)
            # if zip contains one top folder, set WORKDIR to that folder; adjust as needed
            kids = [os.path.join(extract_to, x) for x in os.listdir(extract_to) if not x.startswith(".")]
            dirs = [k for k in kids if os.path.isdir(k) and (pathlib.Path(k) / "scripts" / "train_production_gan_gpu.py").is_file()]
            if len(dirs) == 1:
                WORKDIR = dirs[0]
            print("Extracted; set WORKDIR =", WORKDIR)
            break
    else:
        print("Set REPO_URL to a git URL, or upload a zip to /content/steganography-project.zip and re-run, or mount Drive and set DRIVE_ZIP.")

In [ ]:
# 3) Optional: mount Google Drive to save checkpoints outside Colab (persist after session ends)
USE_DRIVE = False  # set True to mount and copy final checkpoint in last cell

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted.")
else:
    print("Skipping Drive. Enable USE_DRIVE to persist outputs on Drive.")

In [ ]:
# 4) Run production GPU training (same script as on a local machine with CUDA)
import os, subprocess, sys, pathlib

# Ensure WORKDIR points at repo root (folder containing /scripts/…)
if "WORKDIR" not in dir() or not os.path.isfile(pathlib.Path(WORKDIR) / "scripts" / "train_production_gan_gpu.py"):
    WORKDIR = "/content/stego_project"  # adjust to your path
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)

if not os.path.exists("scripts/train_production_gan_gpu.py"):
    raise SystemExit(f"Not found: {os.getcwd()}/scripts/train_production_gan_gpu.py — fix WORKDIR in previous cells.")

EPOCHS = 200
SAMPLES = 8000  # increase if you have time (better generalization; longer runs)
BATCH = 32      # lower to 16 if CUDA OOM (e.g. T4 16GB should be ok for image at default sizes)
MODALITY = "image"  # or "video" or "audio"

cmd = f"{sys.executable} scripts/train_production_gan_gpu.py --modality {MODALITY} --device cuda --epochs {EPOCHS} --train-samples {SAMPLES} --batch {BATCH}"
print(cmd)
r = subprocess.call(cmd, shell=True, cwd=WORKDIR)
if r != 0:
    raise SystemExit(f"Training exited with code {r}")

In [ ]:
# 5) Optional: copy best checkpoint to Google Drive (enable USE_DRIVE in cell 3 first)
if "USE_DRIVE" not in dir():
    USE_DRIVE = False
if "MODALITY" not in dir():
    MODALITY = "image"
if "WORKDIR" not in dir():
    WORKDIR = "/content/stego_project"
if USE_DRIVE:
    import shutil, pathlib, glob
    from pathlib import Path
    ck = Path(WORKDIR) / "models" / "checkpoints" / ("image_gan_improved" if MODALITY=="image" else ("video_gan_improved" if MODALITY=="video" else "audio_gan_improved")) / "best_model.pth"
    if ck.is_file():
        dest = Path("/content/drive/MyDrive/stego_checkpoints")
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ck, dest / ck.name)
        print("Saved to", dest / ck.name)
    else:
        print("Checkpoint not found at", ck)
else:
    print("Download models/checkpoints/ from Colab’s Files pane, or enable USE_DRIVE in cell 3 and re-run this cell.")